In [1]:
import os
from copy import deepcopy
import json
import time
# import xml.etree.ElementTree as ET
# from rich.tree import Tree
# from rich import print as rprint
import io
from typing import List, Union, Tuple, Dict, Optional
from collections.abc import Iterable
from tqdm import tqdm
# import pdfplumber
# import fitz 
import numpy as np
import pandas as pd
import requests
# import xmltodict
import re
from lxml import etree
from pypdf import PdfReader, PdfWriter
from difflib import SequenceMatcher
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.document import DocumentStream
from docling.pipeline.vlm_pipeline import VlmPipeline
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from table_link_to_excel import _curl_get_text, _extract_pmc_info, _try_pmc_direct_table_download, _flatten_columns, sanitize_sheet_name, fetch_html, fetch_pmc_fulltext_xml, pick_table, table_to_dataframe, _clean_text
from bs4 import BeautifulSoup
from advp_formatting_engine import *
from advp_information_retriever import *
from advp_table_extraction import *
from utils import *

/Users/justpqa/advpai/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Test the pipeline by separating into 2 steps: get the table + get the text col

#### Get the table

In [2]:
embeddings_model = AutoModel.from_pretrained("NeuML/pubmedbert-base-embeddings")
embeddings_model_tokenizer = AutoTokenizer.from_pretrained("NeuML/pubmedbert-base-embeddings")

In [3]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]
# test_papers_info_sample = deepcopy(test_papers_info)
# np.random.seed(37)
# np.random.shuffle(test_papers_info_sample)
# test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30820047, "PMC6463297")
# ]
test_papers_info = [
    (20932310, "PMC2964649"), (21123754, "PMC3030225"), (22903471, "PMC3779070"), 
    (23143602, "PMC3510344"), (23535033, "PMC3760995")
]

referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
advp_formatting_engine = ADVPFormattingEngine(referencing_col_df)

referencing_col_require_rag_with_choice_df = pd.read_csv("referencing_cols/ADVP context required col choice.csv")

# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b``
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    print(f"START WORKING WITH {pmid}_{pmcid}")
    # NOTE: find all table id
    found_table = False
    try:
        has_error = table_link_to_excel(pmid, pmcid)
        if has_error:
            print(f"Error in extracting from {pmid}-{pmcid} with table_link_to_excel")
        else:
            found_table = True
            print(f"Success in extracting from {pmid}-{pmcid} with table_link_to_excel")
    except Exception as e:
        print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    # filter for actual GWAS result table: include either SNP (rs....) or CHR, as well as include a p-value cell with p-value col
    curr_intermediate_tables = list(os.listdir("./intermediate_tables"))
    for file_name in curr_intermediate_tables:
        if f"{pmid}_{pmcid}" in file_name:
            if ".xlsx" in file_name:
                df = pd.read_excel(f"./intermediate_tables/{file_name}")
            else:
                df = pd.read_csv(f"./intermediate_tables/{file_name}")
            # check if the table has SNP or CHR, and p value filter
            snp_pattern_1 = re.compile(r"rs\d+")
            snp_pattern_2 = re.compile(r"rsl\d+") # extra pattern of rsl
            snp_mask_1 = df.applymap(lambda x: snp_pattern_1.search(str(x)) is not None)
            snp_mask_2 = df.applymap(lambda x: snp_pattern_2.search(str(x)) is not None)
            snp_filter = snp_mask_1.values.any() or snp_mask_2.values.any()
            chr_pattern_1 = re.compile(r"(?<![A-Za-z.])[Cc][Hh][Rr](?![A-Za-z.])")
            chr_pattern_2 = re.compile('(?<![A-Za-z.])\d+(?![A-Za-z.])')
            chr_mask_1 = df.applymap(lambda x: chr_pattern_1.search(str(x)) is not None)
            chr_mask_2 = df.applymap(lambda x: chr_pattern_2.search(str(x)) is not None)
            chr_filter = chr_mask_1.values.any() and chr_mask_2.values.any()
            p_value_pattern_1 = re.compile(r"\d+\.\d+")
            p_value_pattern_2 = re.compile(r"(?<![A-Za-z])[Pp](?![A-Za-z])",)
            p_value_mask_1 = df.applymap(lambda x: p_value_pattern_1.search(str(x)) is not None)
            p_value_mask_2 = df.applymap(lambda x: p_value_pattern_2.search(str(x)) is not None)
            p_value_filter = p_value_mask_1.values.any() and p_value_mask_2.values.any()
            if (snp_filter or chr_filter) and p_value_filter:
                continue
            else:
                os.remove(f"./intermediate_tables/{file_name}")

    # Temporary remove this part
    # if not found_table:
    #     try: 
    #         df_lst = extract_tables_lst_from_paper(pmcid, f"papers/{pmid}_{pmcid}.pdf")
    #         for i, df in enumerate(df_lst):
    #             if df.shape[0] > 0:
    #                 df.to_csv(f"intermediate_tables/{pmid}_{pmcid}_{i}_from_pdf.csv", index = False)
    #                 found_table = True
    #         if found_table:
    #             print(f"Success in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
    #         else:
    #             print(f"Error in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
    #     except Exception as e:
    #         print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.DataFrame(columns = referencing_col_df["column"].to_list())
        for file_name in os.listdir("intermediate_tables"):
            if str(pmid) in file_name and pmcid in file_name:
                if ".xlsx" in file_name:
                    df = pd.read_excel(f"intermediate_tables/{file_name}")
                else:
                    df = pd.read_csv(f"intermediate_tables/{file_name}")

                # save matching dict for debug
                # Conduct cleaning before matching
                clean_df = advp_formatting_engine.clean_df(df)
                file_name_to_matching[file_name] = advp_formatting_engine.match_many_col_to_ref_col(clean_df)

                harmonized_df = advp_formatting_engine.format_original_table(df, remove_unique_col = True)
                if harmonized_df_all is None:
                    harmonized_df_all = harmonized_df.copy()
                else:
                    harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in harmonizing from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in harmonizing from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        harmonized_df_all["SNP"] = harmonized_df_all["SNP"].apply(lambda x: clean_snp(x))
        int_col = ["Chr"]
        for c in int_col:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_int(x))
        numerical_col_with_many_numbers = ["Effect"]
        for c in numerical_col_with_many_numbers:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: extract_first_number_from_str(x))
        float_col = {"P-value": 15, "Effect": None, "AF": None} # effect back to none since in some test it is 3 digits
        for c in float_col:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_float(x))
            if float_col[c] is not None:
                harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_round(x, float_col[c]))
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in converting to number from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in converting to number from {pmid}-{pmcid} with error {e}")
    print()

    # NOTE: you need this to prevent 429 error of requesting too much => blocked
    time.sleep(30)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


START WORKING WITH 20932310_PMC2964649
Successfully retrieve list of tables' id
Success in extracting from 20932310-PMC2964649 with table_link_to_excel


/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:58: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_1 = df.applymap(lambda x: snp_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_2 = df.applymap(lambda x: snp_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:63: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_1 = df.applymap(lambda x: chr_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_2 = df.applymap(lambda x: chr_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx

Success in harmonizing from 20932310-PMC2964649
Success in converting to number from 20932310-PMC2964649

START WORKING WITH 21123754_PMC3030225
Error in extracting from 21123754-PMC3030225 with error Expecting value: line 1 column 2 (char 1)
Success in harmonizing from 21123754-PMC3030225
Success in converting to number from 21123754-PMC3030225

START WORKING WITH 22903471_PMC3779070
Successfully retrieve list of tables' id
Success in extracting from 22903471-PMC3779070 with table_link_to_excel


/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:58: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_1 = df.applymap(lambda x: snp_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_2 = df.applymap(lambda x: snp_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:63: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_1 = df.applymap(lambda x: chr_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_2 = df.applymap(lambda x: chr_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx

Success in harmonizing from 22903471-PMC3779070
Success in converting to number from 22903471-PMC3779070

START WORKING WITH 23143602_PMC3510344
Successfully retrieve list of tables' id
Success in extracting from 23143602-PMC3510344 with table_link_to_excel


/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:58: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_1 = df.applymap(lambda x: snp_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_2 = df.applymap(lambda x: snp_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:63: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_1 = df.applymap(lambda x: chr_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_2 = df.applymap(lambda x: chr_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx

Success in harmonizing from 23143602-PMC3510344
Success in converting to number from 23143602-PMC3510344

START WORKING WITH 23535033_PMC3760995
Successfully retrieve list of tables' id
Success in extracting from 23535033-PMC3760995 with table_link_to_excel


/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:58: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_1 = df.applymap(lambda x: snp_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  snp_mask_2 = df.applymap(lambda x: snp_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:63: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_1 = df.applymap(lambda x: chr_pattern_1.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_12641/2276687928.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chr_mask_2 = df.applymap(lambda x: chr_pattern_2.search(str(x)) is not None)
/var/folders/5b/jn4p_4vx

Success in harmonizing from 23535033-PMC3760995
Success in converting to number from 23535033-PMC3760995



In [4]:
for f in file_name_to_matching:
    for ref_col in file_name_to_matching[f]:
        for i in range(len(file_name_to_matching[f][ref_col])):
            file_name_to_matching[f][ref_col][i] = (file_name_to_matching[f][ref_col][i][0], float(file_name_to_matching[f][ref_col][i][1]))
with open("test_matching_dict.json", "w") as f:
    json.dump(file_name_to_matching, f, indent=4)

#### Get the text col

In [5]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]
# test_papers_info_sample = deepcopy(test_papers_info)
# np.random.seed(37)
# np.random.shuffle(test_papers_info_sample)
# test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30820047, "PMC6463297")
# ]
test_papers_info = [
    (20932310, "PMC2964649"), (21123754, "PMC3030225"), (22903471, "PMC3779070"), 
    (23143602, "PMC3510344"), (23535033, "PMC3760995")
]

referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
advp_information_retriever = ADVPInformationRetriever(referencing_col_require_rag_df, use_hf = False, device = "mps")

referencing_col_require_rag_with_choice_df = pd.read_csv("referencing_cols/ADVP context required col choice.csv")

# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b``
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    try:
        harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        col_require_rag_to_possible_info = advp_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
    except Exception as e:
        print(f"Error in extracting columns from paper text and LLM from {pmid}-{pmcid} with error {e}")
        col_require_rag_to_possible_info = {col: [] for col in referencing_col_require_rag_df["column"].unique()}

    try:
        threshold = 0.5
        for ref_col, ref_col_choice in zip(
            referencing_col_require_rag_with_choice_df["column"], referencing_col_require_rag_with_choice_df["choice"]
        ):
            if len(col_require_rag_to_possible_info[ref_col]) > 0:
                col_with_category = []
                if ref_col in col_require_rag_to_possible_info:
                    ref_col_choice_lst = ref_col_choice.split(",")
                    detail_choice_similarity = calculate_similarity_scores(col_require_rag_to_possible_info[ref_col], ref_col_choice_lst, embeddings_model, embeddings_model_tokenizer)
                    # get the max of each col
                    max_by_choice = detail_choice_similarity.max(axis = 0).values
                    valid_choice = []
                    for i in range(len(ref_col_choice_lst)):
                        if max_by_choice[i] > threshold:
                            valid_choice.append(ref_col_choice_lst[i])
                    col_require_rag_to_possible_info[f"{ref_col} category"] = deepcopy(valid_choice)
                    col_with_category.append(ref_col)
                for ref_col in col_with_category:
                    temp, temp_category = col_require_rag_to_possible_info[ref_col], col_require_rag_to_possible_info[f"{ref_col} category"]
                    col_require_rag_to_possible_info[ref_col] = deepcopy(temp_category)
                    col_require_rag_to_possible_info[f"{ref_col} details"] = deepcopy(temp)
                    del col_require_rag_to_possible_info[f"{ref_col} category"]
            else:
                col_require_rag_to_possible_info[f"{ref_col} details"] = []
    except Exception as e:
        print(f"Error in extracting columns with choice from {pmid}-{pmcid} with error {e}")
    
    try:
        # for cohort need to do differently
        # col_require_rag_to_possible_info["Cohort"] = col_require_rag_to_possible_info["Cohort"] + gwas_information_retriever_cohort.extract_possible_info_from_paper(pmid, pmcid)
        # col_require_rag_to_possible_info["Cohort"] = list(set([item.lower() for item in col_require_rag_to_possible_info["Cohort"]]))
        print(pmid, pmcid)
        print(col_require_rag_to_possible_info)
        harmonized_df_all = match_possible_info_to_df(harmonized_df_all, col_require_rag_to_possible_info, embeddings_model, embeddings_model_tokenizer)
        # harmonized_df_all = match_possible_info_to_df_with_clues(harmonized_df_all, pmid, pmcid, gwas_information_retriever)
        # harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        # if (pmid, pmcid) in test_papers_info_sample:
        #     harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}_pred.csv", index = False)
        harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in extracting columns from paper text from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in extracting columns from paper text + matching from {pmid}-{pmcid} with error {e}")

20932310 PMC2964649
{'Cohort': ['nia', 'alzheimers disease neuroimaging initiative', 'adni database', 'alzheimers disease cooperative study', 'nibib', 'adni', 'fda'], 'Population': ['Caucasian', 'Multi-ethnic', 'African American', 'Caribbean Hispanic', 'Asian'], 'Stage': ['Discovery', 'Replication', 'Joint-analysis', 'Meta-analysis'], 'Imputation': ['genotype', 'imputation', 'aligator', 'reference panel', 'snp', 'pca', 'smartpca'], 'Study type': ['geno2df', 'genotyping', 'apoe 4 genotype', 'sequencing'], 'Phenotype': ['alzheimers disease', 'mci', 'late-onset alzheimers disease', 'ad'], 'Population details': ['race', 'caucasians', 'ancestry', 'ethnic', 'non-caucasian'], 'Stage details': ['combined', 'meta', 'discovery', 'replication']}
Success in extracting columns from paper text from 20932310-PMC2964649
21123754 PMC3030225
{'Cohort': [], 'Population': [], 'Stage': [], 'Imputation': [], 'Study type': [], 'Phenotype': [], 'Population details': [], 'Stage details': []}
Success in extract

#### Experiment result

- (20932310, "PMC2964649"):
    + Currently estimate many col as P-value due to current pipeline concat all index into 1 single col name, but some col has multi-index that has terms like P-tau but actually are beta terms (for example: Linear regression result without using covariates.5.Normal: A\u03b2 1-42.BETA.Normal: T-tau.Normal: P-tau 181p.MCI: T-tau.MCI: P-tau 181p.AD: A\u03b2 1-42.AD: T-tau.AD: P-tau 181p.Linear regression result using APOE genotype and age as covariates.MCI: A\u03b2 1-42)
- (21123754, "PMC3030225"):
    + Cannot access due to API limitation